# Compiling the final dataframe
provides daily data from 1940 - 2026

In [1]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

In [2]:
PATH = "/Users/kaijz/Desktop/NOAA/data"

# where figs are saved
RESULTS = "/Users/kaijz/Desktop/NOAA/ensoregimes/ensoregimes/results/figs"

# df = pd.read_csv(f'{PATH}/daily_dataframe')

In [3]:

# read in e/c indices (monthly, 1940/01 - 2026/01)
ec = pd.read_csv(f"{PATH}/ecindex_ersstv6.csv")

# read in WR indices (daily, 1940/01/01 - 2025/12/31)
wr = pd.read_csv(f"{PATH}/WR_ERA5_4WRs.csv")

# read in RONI 3.4 (monthly, 1940/01 - idk)
RONI = xr.open_dataset(f'{PATH}/RONI.nc')['RONI']

# read in ONI (seasonal 3-month avg)
## ONI = xr.open_dataset(f'{PATH}/ONI.nc')['ONI']
oni = pd.read_csv(f'{PATH}/ONI_34.csv') # starts at 1950

In [4]:
# reading in roni

roni = pd.DataFrame(RONI, RONI.time, columns=['RONI'])
roni = roni.reset_index()

roni = roni.rename(columns={'index': 'date'})
roni["date"] = pd.to_datetime(roni["date"])
roni["year"] = roni["date"].dt.year
roni["month"] = roni["date"].dt.month

roni = roni.drop(labels='date', axis=1)


In [5]:
# reading in oni, from .ascii file downloaded

oni['date'] = pd.date_range(start="1950-01-01", end="2025-12-31", freq="MS")

oni['month'] = oni['date'].dt.month
oni = oni.rename(columns={'ANOM': 'ONI', 'YR': 'year'})

oni = oni.drop(labels=['SEAS', 'TOTAL', 'date'], axis=1)

In [6]:
# process wr

wr = wr.rename(columns={wr.columns[0]: "date"})
wr["date"] = pd.to_datetime(wr["date"])
wr["year"] = wr["date"].dt.year
wr["month"] = wr["date"].dt.month

In [7]:
df = wr.merge(ec, on=["year", "month"], how="left")
df = df.merge(roni, on=["year", 'month'], how='left')
df = df.merge(oni, on=["year", 'month'], how='left')
df = df.sort_values("date").reset_index(drop=True)

df = df.dropna() # to get rid of 1950

In [8]:
### add the season here
def season_of(m):
    if m in (12, 1, 2): return "DJF"
    if m in (3, 4, 5):  return "MAM"
    if m in (6, 7, 8):  return "JJA"
    return "SON"
df["season"] = df["month"].apply(season_of)

In [9]:
# WR labels ( actual mapping)


# for plotting purposes
WR_NAMES = {0: "Pacific Trough", 1: "Greenland High", 2: "Pacific Ridge", 3:"Alaskan Ridge", 4: "No Regime"}


df["WR_name"] = df["WR"].map(WR_NAMES)

In [10]:
# for plots
WR_SHORT = {0: "PT", 1: "GH", 2: "PR", 3:"AR", 4: "N/A"}
WR_COLORS = {0: "#0b6fe2", 1: "#30b430", 2: "#d33131",3: "#ddf00a", 4: "#5c676f"}
WR_ORDER = [0, 2, 3, 1, 4] # PT # PR #AR # GH # NR

state_order_5 = ["strong La Nina", "weak La Nina", "neutral",
                 "weak El Nino", "strong El Nino"]

state_order_3 = ["La Nina", "neutral", "El Nino"]

In [11]:
## assigning labels to WC classifications: 

print(f"Total daily samples: {len(df):,}")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"\nWR class distribution:")
print(df["WR_name"].value_counts().sort_index())
print('with number indices')
# print(df["WR"].value_counts().sort_index())

Total daily samples: 27,740
Date range: 1950-01-01 to 2025-12-31

WR class distribution:
WR_name
Alaskan Ridge     5336
Greenland High    5971
No Regime         4151
Pacific Ridge     5434
Pacific Trough    6848
Name: count, dtype: int64
with number indices


In [12]:
# ============================================================
# 1. Event dictionary from DJF-mean E and C
# ============================================================

# Use ec (monthly), aggregate to DJF means (winter year = year of Jan/Feb)
ec_djf = ec.copy()
ec_djf["winter_year"] = np.where(ec_djf["month"] == 12,
                                 ec_djf["year"] + 1, ec_djf["year"])
ec_djf = (ec_djf[ec_djf["month"].isin([12, 1, 2])]
          .groupby("winter_year")[["E_index", "C_index"]].mean())

THRESH = 1.0   # DJF-mean threshold (sigma)
EXTREME = 2  # for the "name these on the plot" bucket

def classify_event(row, thresh=THRESH):
    E, C = row["E_index"], row["C_index"]
    if max(abs(E), abs(C)) < thresh:
        return "neutral"
    if abs(E) >= abs(C):
        return "E+" if E > 0 else "E-"
    return "C+" if C > 0 else "C-"

ec_djf["event_type"] = ec_djf.apply(classify_event, axis=1)
ec_djf["is_extreme"] = ec_djf[["E_index", "C_index"]].abs().max(axis=1) >= EXTREME

# Build dictionary, optionally restricting to extreme ones
events_all     = ec_djf["event_type"].to_dict()
events_extreme = ec_djf[ec_djf["is_extreme"]]["event_type"].to_dict()

print("Event-type counts (DJF threshold = 1.0):")
print(ec_djf["event_type"].value_counts())
print(f"\nExtreme events (|E| or |C| >= 2.0):")
print(ec_djf[ec_djf["is_extreme"]].sort_values("winter_year"))



Event-type counts (DJF threshold = 1.0):
event_type
neutral    45
C-         23
C+         12
E+          4
E-          3
Name: count, dtype: int64

Extreme events (|E| or |C| >= 2.0):
              E_index   C_index event_type  is_extreme
winter_year                                           
1943        -0.412887 -2.042055         C-        True
1974        -0.550814 -2.691872         C-        True
1976        -0.523016 -2.441637         C-        True
1983         3.341567  0.245839         E+        True
1989        -0.094997 -2.468244         C-        True
1998         3.703744  0.343347         E+        True
1999         0.352949 -2.354258         C-        True
2000        -0.344817 -2.031170         C-        True
2008        -0.511238 -2.063967         C-        True


In [22]:
SEASONS = {
    1:  "DJF", 2:  "JFM", 3:  "FMA", 4:  "MAM",
    5:  "AMJ", 6:  "MJJ", 7:  "JJA", 8:  "JAS",
    9:  "ASO", 10: "SON", 11: "OND", 12: "NDJ",
}

def enso_state(data, metric, seasonal, thresh=0.5):
    '''

    data : pd.DataFrame
        Must contain columns [metric, 'month']
    metric: {'ONI', RONI'}
    seasonal: boolean, True if 3-class scheme w/seasonally varying percentile thresholds is to be used

    returns dataframe with 'METRIC_state' calculated that way

    '''

    if metric not in ['RONI', 'ONI']:
        raise ValueError('Please enter RONI/ONI metric.')
    
    df = data.copy()
    col = f"{metric}_state"
    df[col] = "neutral"

    if seasonal:
        df["season"] = df["month"].map(SEASONS)
        # compute 25/75th percentile per season on the metric values # using unique month-level values so daily duplicates don't bias quantiles
        monthly = df.groupby(["year", "month"], as_index=False)[metric].first()
        monthly["season"] = monthly["month"].map(SEASONS)
        thresholds = (monthly.groupby("season")[metric]
                             .quantile([0.25, 0.75])
                             .unstack())
        thresholds.columns = ["p25", "p75"]

        for szn, row in thresholds.iterrows():
            mask = df["season"] == szn
            df.loc[mask & (df[metric] >= row["p75"]), col] = "El Nino"
            df.loc[mask & (df[metric] <= row["p25"]), col] = "La Nina"

    else:
        df.loc[df[metric] >=  thresh*2, col] = "strong El Nino"
        df.loc[(df[metric] >=  thresh) & (df[metric] <  thresh*2), col] = "weak El Nino"
        df.loc[(df[metric] <= -thresh) & (df[metric] > -thresh*2), col] = "weak La Nina"
        df.loc[df[metric] <= -thresh*2, col] = "strong La Nina"

    return df

    

state_colors = {
    "strong La Nina": "#2166ac",
    "weak La Nina":   "#92c5de",
    "neutral":        "#dddddd",
    "weak El Nino":   "#f4a582",
    "strong El Nino": "#b2182b",
}

state_order = ["strong La Nina","weak La Nina", "neutral","weak El Nino","strong El Nino"]

# add 3-bin seasonal ONI count
df = enso_state(df, 'ONI', seasonal=True)

# add 5-bin RONI state count
df = enso_state(df, 'RONI', seasonal=False)

# print("ENSO state daily counts:")
# print(df["roni_state"].value_counts().reindex(state_order))
# print(df["oni_state"].value_counts().reindex(state_order))

In [14]:
df["decade"] = (df["year"] // 10) * 10

In [23]:
df

,date,WR,distances,year,month,E_index,C_index,RONI,ONI,season,WR_name,decade,ONI_state,RONI_state
3650,1950-01-01,2,71.367786,1950,1,-1.407605,-1.494195,-1.160015,-1.53,DJF,Pacific Ridge,1950,La Nina,strong La Nina
3651,1950-01-02,2,69.577674,1950,1,-1.407605,-1.494195,-1.160015,-1.53,DJF,Pacific Ridge,1950,La Nina,strong La Nina
3652,1950-01-03,2,67.044385,1950,1,-1.407605,-1.494195,-1.160015,-1.53,DJF,Pacific Ridge,1950,La Nina,strong La Nina
3653,1950-01-04,2,64.494670,1950,1,-1.407605,-1.494195,-1.160015,-1.53,DJF,Pacific Ridge,1950,La Nina,strong La Nina
3654,1950-01-05,2,62.762696,1950,1,-1.407605,-1.494195,-1.160015,-1.53,DJF,Pacific Ridge,1950,La Nina,strong La Nina
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31385,2025-12-27,4,63.899119,2025,12,-0.788619,-0.270784,-1.030602,-0.54,NDJ,No Regime,2020,neutral,strong La Nina
31386,2025-12-28,1,57.189714,2025,12,-0.788619,-0.270784,-1.030602,-0.54,NDJ,Greenland High,2020,neutral,strong La Nina
31387,2025-12-29,1,48.719027,2025,12,-0.788619,-0.270784,-1.030602,-0.54,NDJ,Greenland High,2020,neutral,strong La Nina
31388,2025-12-30,1,46.044778,2025,12,-0.788619,-0.270784,-1.030602,-0.54,NDJ,Greenland High,2020,neutral,strong La Nina


In [28]:
df.to_csv(f"{PATH}/daily_dataframe.csv", index=False) # f"{RESULTS}/ONIvRONI/{timerange}.png"

In [24]:
data = df

data = data["RONI_state"].value_counts().reindex(state_order, fill_value=0)

In [ ]:
# # Long format for seaborn so we can order categories
# def long_state_counts(data):

#     data = enso_state(data, "RONI", seasonal=True)

    
#     counts_roni = data["roni_state"].value_counts().reindex(state_order, fill_value=0)
#     counts_oni  = data["oni_state"].value_counts().reindex(state_order, fill_value=0)
#     out = pd.DataFrame({
#         "state": list(state_order) * 2,
#         "count": list(counts_roni.values) + list(counts_oni.values),
#         "index": ["RONI"] * 5 + ["ONI"] * 5,
#     })
#     return out


# def plt_hist_time(timerange, data):
#     timerange = str(timerange)
#     fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 4.5),
#                                    gridspec_kw={"width_ratios": [1, 2]})

#     # --- Left: ordered bar plot ---
#     plot_df = long_state_counts(data)
#     sns.barplot(
#         data=plot_df, x="state", y="count", hue="index",
#         order=state_order, palette={"RONI": "#cb181d", "ONI": "#2c7fb8"},
#         edgecolor="black", alpha=0.7, ax=ax1,
#     )
#     ax1.set_title(f"ENSO state frequency, {timerange}s")
#     ax1.set_xlabel(""); ax1.set_ylabel("Daily count")
#     ax1.tick_params(axis="x", rotation=30)
#     ax1.legend(title="Index")

#     # --- Right: ONI vs RONI time series for this time range ---
#     # Use monthly resolution (one point per month) so the line is clean
#     monthly = (data.groupby(["year", "month"])[["ONI", "RONI"]]
#                    .first().reset_index())
#     monthly["date"] = pd.to_datetime(monthly[["year", "month"]].assign(day=15))
#     monthly = monthly.sort_values("date")

#     ax2.plot(monthly["date"], monthly["ONI"],  color="#2c7fb8", lw=1.5, label="ONI")
#     ax2.plot(monthly["date"], monthly["RONI"], color="#cb181d", lw=1.5, label="RONI")
#     # Fill where they differ
#     ax2.fill_between(monthly["date"], monthly["ONI"], monthly["RONI"],
#                      where=monthly["RONI"] > monthly["ONI"],
#                      color="#cb181d", alpha=0.2, label="RONI > ONI")
#     ax2.fill_between(monthly["date"], monthly["ONI"], monthly["RONI"],
#                      where=monthly["RONI"] < monthly["ONI"],
#                      color="#2c7fb8", alpha=0.2, label="RONI < ONI")
#     # Threshold lines
#     for h in [-1.0, -0.5, 0.5, 1.0]:
#         ax2.axhline(h, color="gray", lw=0.5, ls=":", alpha=0.6)
#     ax2.axhline(0, color="k", lw=0.6)

#     # Annotate extreme E/C events in this decade
#     for wy, etype in events_extreme.items():
#         event_date = pd.Timestamp(f"{wy}-01-15")  # mid-DJF anchor
#         if monthly["date"].min() <= event_date <= monthly["date"].max():
#             color = "#b2182b" if etype.startswith("E") else "#f4a582"
#             ax2.axvline(event_date, color=color, ls="--", lw=1, alpha=0.8)
#             ax2.text(event_date, ax2.get_ylim()[1] * 0.92,
#                      f"{wy} {etype}", rotation=90, va="top", ha="right",
#                      fontsize=7.5, color=color)

#     ax2.set_title(f"ONI vs RONI, {timerange}s  (dashed = extreme E/C DJF events)")
#     ax2.set_xlabel(""); ax2.set_ylabel("Index value")
#     ax2.legend(loc="upper right", fontsize=8, ncol=2)

#     plt.tight_layout()
#     plt.savefig((f"{RESULTS}/ONIvRONI/{timerange}.png"))
#     plt.show()

# for decade, data in df.groupby("decade"):
#     print('decade is ', decade, type(decade))
#     plt_hist_time(decade, data)
